In [ ]:
#libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [ ]:
#load dataset
df=pd.read_csv("/content/drive/MyDrive/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.head())
print(df.info())

In [ ]:
#check null values
print(df.isnull().sum())

In [ ]:
#no neeed to use dropna and fullna
#now checking duplicates
print(df.duplicated().sum())
print("shape of dataset:",df.shape)

In [ ]:
#finding outliers
#IQR method
num_cols=df.select_dtypes(include=np.number).columns
for col in num_cols:
  Q1=df[col].quantile(0.25)
  Q3=df[col].quantile(0.75)
  IQR=Q3-Q1
  lower_limit=Q1-1.5*IQR
  upper_limit=Q3+1.5*IQR
  outliers = df[(df[col]<lower_limit)|(df[col]>upper_limit)]
  print(f"{col}:{len(outliers)} rows detected as outliers")

In [ ]:
#now we have to remove outliers from senior citizen column
num_cols=df.select_dtypes(include=np.number).columns
for col in num_cols:
  Q1=df[col].quantile(0.25)
  Q3=df[col].quantile(0.75)
  IQR=Q3-Q1
  lower_limit=Q1-1.5*IQR
  upper_limit=Q3+1.5*IQR
  df=df[(df[col]>=lower_limit)&(df[col]<=upper_limit)]
print("shape after removing outliers:",df.shape)

In [ ]:
le=LabelEncoder()
cat_cols=df.select_dtypes(include='object').columns
for col in cat_cols:
  df[col]=le.fit_transform(df[col])
print(df.head())

In [ ]:
#input and output split
X=df.drop("Churn",axis=1)#feature
Y=df["Churn"]
print("input columns:",X.head())


In [ ]:
#train and test
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(
    X,Y,test_size=0.2,random_state=42)

In [ ]:
#featurescaling
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.transform(X_test)
print(X_train)

In [ ]:
#model selction
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC()
}

for name, model in models.items():
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)
    acc = (Y_pred == Y_test).mean()#find fraction of all true values
    print(f"{name}: {acc:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score,recall_score,f1_score

cm = confusion_matrix(Y_test, Y_pred)
print(cm)
print(classification_report(Y_test, Y_pred))  # precision, recall, f1, support
print(accuracy_score(Y_test, Y_pred))  # accuracy

In [47]:
# ============================================================
# PREDICT A NEW CUSTOMER USING YOUR TRAINED SVM MODEL
# ============================================================

import pandas as pd
import numpy as np

# New customer details (raw values, just like original dataset)
new_customer = pd.DataFrame([{
    'customerID'       : 'TEST-001',
    'gender'           : 'Male',
    'SeniorCitizen'    : 0,
    'Partner'          : 'No',
    'Dependents'       : 'No',
    'tenure'           : 2,
    'PhoneService'     : 'Yes',
    'MultipleLines'    : 'No',
    'InternetService'  : 'Fiber optic',
    'OnlineSecurity'   : 'No',
    'OnlineBackup'     : 'No',
    'DeviceProtection' : 'No',
    'TechSupport'      : 'No',
    'StreamingTV'      : 'Yes',
    'StreamingMovies'  : 'Yes',
    'Contract'         : 'Month-to-month',
    'PaperlessBilling' : 'Yes',
    'PaymentMethod'    : 'Electronic check',
    'MonthlyCharges'   : 85.50,
    'TotalCharges'     : '171.00'
}])

# Step 1: Drop customerID (same as training)
new_customer = new_customer.drop('customerID', axis=1)

# Step 2: Apply LabelEncoder (must use same le object from training)
cat_cols = new_customer.select_dtypes(include='object').columns
for col in cat_cols:
    new_customer[col] = le.transform(new_customer[col])

# Step 3: Apply StandardScaler (must use same sc object from training)
new_customer_scaled = sc.transform(new_customer)

# Step 4: Predict using SVM
svm_model = models['SVM']
prediction = svm_model.predict(new_customer_scaled)

# Step 5: Show result
result = 'YES - Customer will CHURN 🚨' if prediction[0] == 1 else 'NO - Customer will STAY ✅'
print("Churn Prediction:", result)

ValueError: y contains previously unseen labels: 'Male'